# PrismML Bonsai 2 27B — Colab FastEval-50 + tokens/s

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/PrismML_Ternary_Bonsai2_27B_FastEval50_Colab.ipynb)

This notebook tests **`prism-ml/Ternary-Bonsai-2-27B-gguf`**, derived from **Qwen3.8-27B**, using PrismML's required **custom llama.cpp fork**.

### What it measures
- **Official llama-bench throughput:** prompt processing (`pp512`) and token generation (`tg128`) in **tokens/s**.
- **Live server throughput:** prompt tokens/s and generated tokens/s from llama.cpp native timing data.
- **FastEval-50 quality:** 50 deterministic multiple-choice questions:
  - 10 × ARC-Challenge
  - 10 × BoolQ
  - 10 × OpenBookQA
  - 10 × HellaSwag
  - 10 × WinoGrande
- Per-benchmark accuracy, overall accuracy, latency, token counts, and CSV/JSON export.

### Default model packing
The notebook defaults to **PTQ1_0 (~5.95 GB)** because it is the smallest Bonsai 2 pack and is especially attractive on memory-constrained GPUs. You can switch to **PQ2_0 (~7.21 GB)** in the configuration cell.

> **Important:** stock llama.cpp must not be used for Bonsai 2. The rotated ternary weights require PrismML's Hadamard-aware runtime.

For the fast 50-question evaluation, thinking/reasoning is disabled so the model returns only the requested option label.


In [ ]:
#@title 1. Configuration + GPU check
import os, sys, json, time, pathlib, subprocess, platform

MODEL_REPO = "prism-ml/Ternary-Bonsai-2-27B-gguf"
PACKING = "PTQ1_0" #@param ["PTQ1_0", "PQ2_0"]
MODEL_FILES = {
    "PTQ1_0": "Ternary-Bonsai-2-27B-PTQ1_0.gguf",
    "PQ2_0":  "Ternary-Bonsai-2-27B-PQ2_0.gguf",
}
MODEL_FILE = MODEL_FILES[PACKING]

N_PER_BENCH = 10
SEED = 20260919
CTX_SIZE = 4096
GPU_LAYERS = 99
SERVER_PORT = 8080
RESULT_DIR = pathlib.Path("/content/prism_bonsai2_fasteval50")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_REPO)
print("Packing:", PACKING)
print("FastEval:", N_PER_BENCH * 5, "questions")
print("Python:", sys.version.split()[0], "| Platform:", platform.platform())

if subprocess.run(["bash","-lc","command -v nvidia-smi"],capture_output=True).returncode != 0:
    raise RuntimeError("No NVIDIA GPU found. In Colab choose Runtime → Change runtime type → GPU.")

subprocess.run(["nvidia-smi"], check=True)


In [ ]:
#@title 2. Install dependencies + build PrismML llama.cpp fork (CUDA)
import os, sys, subprocess, pathlib, torch

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "huggingface_hub>=0.34", "hf_xet", "datasets>=3.0", "pandas", "requests", "tqdm"],
    check=True,
)

LLAMA_DIR = pathlib.Path("/content/prism-llama.cpp")
if LLAMA_DIR.exists():
    subprocess.run(["git","-C",str(LLAMA_DIR),"fetch","--depth","1","origin"], check=True)
    subprocess.run(["git","-C",str(LLAMA_DIR),"reset","--hard","origin/master"], check=True)
else:
    subprocess.run(
        ["git","clone","--depth","1","https://github.com/PrismML-Eng/llama.cpp.git",str(LLAMA_DIR)],
        check=True,
    )

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available in this runtime.")

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
cuda_arch = f"{major}{minor}"
print("GPU:", gpu_name, "| CUDA capability:", f"{major}.{minor}", "| CMake arch:", cuda_arch)

BUILD_DIR = LLAMA_DIR / "build"
cmake_cmd = [
    "cmake", "-S", str(LLAMA_DIR), "-B", str(BUILD_DIR),
    "-DGGML_CUDA=ON",
    "-DGGML_NATIVE=OFF",
    f"-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}",
    "-DCMAKE_BUILD_TYPE=Release",
]
subprocess.run(cmake_cmd, check=True)
subprocess.run(
    ["cmake","--build",str(BUILD_DIR),"--config","Release","-j","2",
     "--target","llama-server","llama-bench"],
    check=True,
)

LLAMA_SERVER = BUILD_DIR / "bin" / "llama-server"
LLAMA_BENCH = BUILD_DIR / "bin" / "llama-bench"
assert LLAMA_SERVER.exists(), LLAMA_SERVER
assert LLAMA_BENCH.exists(), LLAMA_BENCH
print("✓ PrismML runtime built")
print(" server:", LLAMA_SERVER)
print(" bench :", LLAMA_BENCH)


In [ ]:
#@title 3. Download Bonsai 2 27B GGUF from Hugging Face
from huggingface_hub import hf_hub_download
import pathlib, os

MODEL_DIR = pathlib.Path("/content/models/prism-bonsai2-27b")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = pathlib.Path(hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=str(MODEL_DIR),
))

size_gib = MODEL_PATH.stat().st_size / (1024**3)
print("✓ Model ready:", MODEL_PATH)
print(f"File size: {size_gib:.2f} GiB")


In [ ]:
#@title 4. llama-bench — pp512 + tg128 tokens/s
import subprocess, json, pandas as pd, re, pathlib
from IPython.display import display

bench_cmd = [
    str(LLAMA_BENCH),
    "-m", str(MODEL_PATH),
    "-ngl", str(GPU_LAYERS),
    "-p", "512",
    "-n", "128",
    "-r", "3",
    "-o", "json",
]
print("Running:", " ".join(bench_cmd))
bench = subprocess.run(bench_cmd, text=True, capture_output=True)

(RESULT_DIR / "llama_bench_stdout.txt").write_text(bench.stdout)
(RESULT_DIR / "llama_bench_stderr.txt").write_text(bench.stderr)

if bench.returncode != 0:
    print(bench.stdout)
    print(bench.stderr[-5000:])
    raise RuntimeError(f"llama-bench failed with exit code {bench.returncode}")

raw = bench.stdout.strip()
try:
    bench_json = json.loads(raw)
except json.JSONDecodeError:
    # Some builds print a small prefix/suffix around JSON. Recover the outer JSON array/object.
    starts = [p for p in (raw.find("["), raw.find("{")) if p >= 0]
    start = min(starts) if starts else -1
    end = max(raw.rfind("]"), raw.rfind("}"))
    if start < 0 or end <= start:
        print(raw)
        raise
    bench_json = json.loads(raw[start:end+1])

rows = bench_json if isinstance(bench_json, list) else [bench_json]
bench_df = pd.DataFrame(rows)

preferred = [c for c in [
    "model_type","model_size","model_n_params","backend",
    "n_batch","n_ubatch","n_threads","n_gpu_layers",
    "n_prompt","n_gen","avg_ts","stddev_ts"
] if c in bench_df.columns]
display(bench_df[preferred] if preferred else bench_df)

bench_df.to_csv(RESULT_DIR / "llama_bench.csv", index=False)
with open(RESULT_DIR / "llama_bench.json","w") as f:
    json.dump(bench_json, f, indent=2)

for _, r in bench_df.iterrows():
    pp = int(r.get("n_prompt", 0) or 0)
    tg = int(r.get("n_gen", 0) or 0)
    ts = r.get("avg_ts", None)
    if ts is not None:
        if pp and not tg:
            print(f"PROMPT PROCESSING pp{pp}: {float(ts):.2f} tokens/s")
        elif tg:
            print(f"TOKEN GENERATION tg{tg}: {float(ts):.2f} tokens/s")

print("✓ Saved benchmark results to", RESULT_DIR)


In [ ]:
#@title 5. Start local llama-server (thinking OFF for fast evaluation)
import subprocess, time, requests, pathlib, os, signal

SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"
SERVER_LOG = RESULT_DIR / "llama_server.log"

# Stop a server created by an earlier execution of this cell.
if "server_proc" in globals() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_proc.kill()

log_handle = open(SERVER_LOG, "w")
server_cmd = [
    str(LLAMA_SERVER),
    "-m", str(MODEL_PATH),
    "-c", str(CTX_SIZE),
    "-ngl", str(GPU_LAYERS),
    "--host", "127.0.0.1",
    "--port", str(SERVER_PORT),
    "--jinja",
    "--reasoning-budget", "0",
]
print("Starting:", " ".join(server_cmd))
server_proc = subprocess.Popen(
    server_cmd,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
)

ready = False
for _ in range(180):
    if server_proc.poll() is not None:
        break
    try:
        rr = requests.get(SERVER_URL + "/health", timeout=1)
        if rr.ok:
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if not ready:
    log_handle.flush()
    print(SERVER_LOG.read_text()[-8000:])
    raise RuntimeError("llama-server did not become ready.")

model_info = requests.get(SERVER_URL + "/v1/models", timeout=20).json()
served_model = model_info["data"][0]["id"]
print("✓ Server ready:", SERVER_URL)
print("Served model:", served_model)

subprocess.run(["nvidia-smi"], check=True)


In [ ]:
#@title 6. Live native throughput measurement — prompt tok/s + generation tok/s
import requests, json, pandas as pd
from IPython.display import display

speed_payload = {
    "prompt": (
        "Write a concise technical explanation of why low-bit neural-network weights can "
        "reduce memory bandwidth pressure during autoregressive inference. "
        "Discuss both benefits and one limitation."
    ),
    "n_predict": 128,
    "temperature": 0.0,
    "cache_prompt": False,
}
speed_resp = requests.post(SERVER_URL + "/completion", json=speed_payload, timeout=600)
speed_resp.raise_for_status()
speed_data = speed_resp.json()
timings = speed_data.get("timings", {})

live_speed = {
    "prompt_tokens": timings.get("prompt_n"),
    "prompt_tokens_per_s": timings.get("prompt_per_second"),
    "generated_tokens": timings.get("predicted_n"),
    "generation_tokens_per_s": timings.get("predicted_per_second"),
    "prompt_ms": timings.get("prompt_ms"),
    "generation_ms": timings.get("predicted_ms"),
}
display(pd.DataFrame([live_speed]))

print("Live output preview:")
print(speed_data.get("content","")[:1200])

with open(RESULT_DIR / "live_speed.json","w") as f:
    json.dump({"metrics": live_speed, "raw_timings": timings}, f, indent=2)

print("\nTOKEN/S SUMMARY")
print("Prompt processing :", live_speed["prompt_tokens_per_s"], "tok/s")
print("Token generation  :", live_speed["generation_tokens_per_s"], "tok/s")


In [ ]:
#@title 7. Build deterministic FastEval-50 dataset (5 benchmarks × 10)
from datasets import load_dataset
import random, pandas as pd

def sample_rows(ds, n, seed):
    n = min(n, len(ds))
    idx = random.Random(seed).sample(range(len(ds)), n)
    return [ds[i] for i in idx]

def mc_prompt(stem, labels, options):
    choices = "\n".join(f"{lab}. {opt}" for lab, opt in zip(labels, options))
    return (
        f"{stem.strip()}\n\n{choices}\n\n"
        f"Answer with only one of these labels: {', '.join(labels)}."
    )

eval_items = []

# ARC-Challenge
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="validation")
for x in sample_rows(arc, N_PER_BENCH, SEED + 1):
    labels = [str(v) for v in x["choices"]["label"]]
    options = [str(v) for v in x["choices"]["text"]]
    eval_items.append({
        "benchmark": "ARC-Challenge",
        "prompt": mc_prompt(str(x["question"]), labels, options),
        "labels": labels,
        "gold": str(x["answerKey"]),
    })

# BoolQ
boolq = load_dataset("google/boolq", split="validation")
for x in sample_rows(boolq, N_PER_BENCH, SEED + 2):
    labels = ["A", "B"]
    stem = f'Passage: {x["passage"]}\n\nQuestion: {x["question"]}'
    eval_items.append({
        "benchmark": "BoolQ",
        "prompt": mc_prompt(stem, labels, ["Yes", "No"]),
        "labels": labels,
        "gold": "A" if bool(x["answer"]) else "B",
    })

# OpenBookQA
obqa = load_dataset("allenai/openbookqa", "main", split="validation")
for x in sample_rows(obqa, N_PER_BENCH, SEED + 3):
    labels = [str(v) for v in x["choices"]["label"]]
    options = [str(v) for v in x["choices"]["text"]]
    eval_items.append({
        "benchmark": "OpenBookQA",
        "prompt": mc_prompt(str(x["question_stem"]), labels, options),
        "labels": labels,
        "gold": str(x["answerKey"]),
    })

# HellaSwag
hs = load_dataset("Rowan/hellaswag", split="validation")
for x in sample_rows(hs, N_PER_BENCH, SEED + 4):
    labels = ["A", "B", "C", "D"]
    stem = (str(x.get("ctx_a","")) + " " + str(x.get("ctx_b",""))).strip()
    options = [str(v) for v in x["endings"]]
    gold_i = int(x["label"])
    eval_items.append({
        "benchmark": "HellaSwag",
        "prompt": mc_prompt(stem, labels, options),
        "labels": labels,
        "gold": labels[gold_i],
    })

# WinoGrande
wg = load_dataset("allenai/winogrande", "winogrande_xl", split="validation")
for x in sample_rows(wg, N_PER_BENCH, SEED + 5):
    labels = ["A", "B"]
    sentence = str(x["sentence"]).replace("_", "_____")
    eval_items.append({
        "benchmark": "WinoGrande",
        "prompt": mc_prompt(sentence, labels, [str(x["option1"]), str(x["option2"])]),
        "labels": labels,
        "gold": "A" if str(x["answer"]) == "1" else "B",
    })

assert len(eval_items) == 50, len(eval_items)
print("✓ FastEval-50 ready")
print(pd.Series([x["benchmark"] for x in eval_items]).value_counts().sort_index())
print("\nExample:\n", eval_items[0]["prompt"][:1200], "\nGold:", eval_items[0]["gold"])


In [ ]:
#@title 8. Run FastEval-50 — accuracy + latency + end-to-end token rate
import requests, time, re, pandas as pd, json, numpy as np
from IPython.display import display

SYSTEM = (
    "You are taking a multiple-choice benchmark. "
    "Return ONLY the requested option label. Do not explain your answer."
)

def parse_label(text, allowed):
    text = (text or "").strip().upper()
    # Prefer an exact short response.
    compact = re.sub(r"[^A-Z0-9]", "", text)
    if compact in allowed:
        return compact
    # Otherwise take the first standalone allowed label.
    for lab in allowed:
        if re.search(rf"(?<![A-Z0-9]){re.escape(lab)}(?![A-Z0-9])", text):
            return lab
    return ""

def ask_mc(item):
    payload = {
        "model": served_model,
        "messages": [
            {"role":"system","content":SYSTEM},
            {"role":"user","content":item["prompt"]},
        ],
        "temperature": 0.0,
        "max_tokens": 8,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    t0 = time.perf_counter()
    r = requests.post(SERVER_URL + "/v1/chat/completions", json=payload, timeout=600)
    latency = time.perf_counter() - t0
    r.raise_for_status()
    d = r.json()
    msg = d["choices"][0]["message"]
    text = msg.get("content") or ""
    usage = d.get("usage") or {}
    pred = parse_label(text, item["labels"])
    completion_tokens = int(usage.get("completion_tokens") or 0)
    prompt_tokens = int(usage.get("prompt_tokens") or 0)
    return {
        "prediction": pred,
        "raw_output": text,
        "correct": pred == item["gold"],
        "latency_s": latency,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        # This is end-to-end request throughput, NOT the pure decode speed.
        "e2e_completion_tok_s": completion_tokens / latency if completion_tokens else np.nan,
    }

records = []
t_all = time.perf_counter()

for i, item in enumerate(eval_items, 1):
    out = ask_mc(item)
    row = {
        "id": i,
        "benchmark": item["benchmark"],
        "gold": item["gold"],
        **out,
    }
    records.append(row)
    mark = "✓" if row["correct"] else "✗"
    print(
        f'{i:02d}/50 {item["benchmark"]:<14} '
        f'gold={row["gold"]:<2} pred={row["prediction"] or "?":<2} '
        f'{mark}  {row["latency_s"]:.2f}s'
    )

total_s = time.perf_counter() - t_all
results_df = pd.DataFrame(records)

summary_df = (
    results_df.groupby("benchmark")
    .agg(
        questions=("correct","size"),
        correct=("correct","sum"),
        accuracy=("correct","mean"),
        avg_latency_s=("latency_s","mean"),
        median_latency_s=("latency_s","median"),
    )
)
summary_df.loc["OVERALL"] = {
    "questions": len(results_df),
    "correct": int(results_df["correct"].sum()),
    "accuracy": float(results_df["correct"].mean()),
    "avg_latency_s": float(results_df["latency_s"].mean()),
    "median_latency_s": float(results_df["latency_s"].median()),
}

display(summary_df.style.format({
    "accuracy":"{:.1%}",
    "avg_latency_s":"{:.2f}",
    "median_latency_s":"{:.2f}",
}))

overall_acc = results_df["correct"].mean()
print(f"\nOVERALL: {results_df['correct'].sum()}/50 = {overall_acc:.1%}")
print(f"Wall time: {total_s:.1f}s | {50/total_s:.3f} questions/s")
print(f"Mean request latency: {results_df['latency_s'].mean():.2f}s")

results_df.to_csv(RESULT_DIR / "fasteval50_results.csv", index=False)
summary_df.to_csv(RESULT_DIR / "fasteval50_summary.csv")

final_summary = {
    "model_repo": MODEL_REPO,
    "model_file": MODEL_FILE,
    "packing": PACKING,
    "questions": 50,
    "correct": int(results_df["correct"].sum()),
    "accuracy": float(overall_acc),
    "wall_time_s": total_s,
    "questions_per_s": 50 / total_s,
    "mean_latency_s": float(results_df["latency_s"].mean()),
    "live_prompt_tokens_per_s": live_speed.get("prompt_tokens_per_s"),
    "live_generation_tokens_per_s": live_speed.get("generation_tokens_per_s"),
}
with open(RESULT_DIR / "fasteval50_report.json","w") as f:
    json.dump(final_summary, f, indent=2)

print("✓ Results saved to", RESULT_DIR)


In [ ]:
#@title 9. Final report + download CSV/JSON
import json, pathlib
from IPython.display import display, Markdown

print("=" * 80)
print("PRISMML BONSAI 2 27B — FASTEVAL-50")
print("=" * 80)
print("Model       :", MODEL_REPO)
print("GGUF        :", MODEL_FILE)
print("Packing     :", PACKING)
print("GPU         :", gpu_name)
print("Accuracy    :", f"{final_summary['correct']}/50 = {final_summary['accuracy']:.1%}")
print("Prompt tok/s:", final_summary["live_prompt_tokens_per_s"])
print("Gen tok/s   :", final_summary["live_generation_tokens_per_s"])
print("Mean latency:", f"{final_summary['mean_latency_s']:.2f}s")
print("Results dir :", RESULT_DIR)

print("\nFiles:")
for p in sorted(RESULT_DIR.iterdir()):
    print(" -", p.name)

try:
    from google.colab import files
    print("\nUse these commands if you want to download the reports:")
    print("files.download(str(RESULT_DIR / 'fasteval50_results.csv'))")
    print("files.download(str(RESULT_DIR / 'fasteval50_report.json'))")
except Exception:
    pass
